In [ ]:
import openai
import gradio as gr
from langchain import LLMChain, OpenAI, PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

import os
import re
os.environ['OPENAI_API_KEY'] = 'Enter OpenAI Key'

### Multi-Agent Combination Optimization
1. 優化策略：流水線方式建構一個元件的推薦流程
2. 降價策略：查看清單，選一個商品做降價

### 資訊傳遞方式

目前使用：component_dict(json) 作為傳遞資料的格式

```python
{'品項': {'reason': '推薦原因', 'name': '產品名稱', 'price': '價格'}}
### 範例
{'Mother board': {'reason': '支援最新的AMD CPU與PCIe 4.0，適合高效能遊戲需求。',
  'name': 'ASUS ROG Strix B550-F Gaming',
  'price': '5999'},
 'Case': {'reason': '這款機殼擁有優秀的散熱性能，且設計美觀，適合高效能遊戲組合。',
  'name': 'Cooler Master MasterBox Q300L',
  'price': '1890'}
  ...}
```

In [2]:
class Agent():
    
    def __init__(self, agent_name, prompt, model, require, budget):
        
        self.agent_name = agent_name
        self.memory = []
        self.require = require
        self.budget = budget
        
        template = ChatPromptTemplate.from_template(prompt)
        model = ChatOpenAI(model=model, temperature=0.7)
        parser = StrOutputParser()

        self.llm = template | model | parser
        
    def receive_message(self, messages):
        self.memory += messages
        
    def send_message(self, content, receive_obj):
        pass
    
    def action(self, component_dict):
        
        component_list = self.get_component_list(component_dict)
        
        user_message = {}
        user_message['component_list'] = component_list
        user_message['require'] = self.require
        user_message['budget'] = self.budget
        
        message = self.llm.invoke(user_message)
        self.memory.append(message)
        component_dict[self.agent_name] = self.parser_message(message)
        
        return component_dict
        
    def check_target(self, target):
        pass
    
    def end_stage(self):
        pass
    
    def get_component_list(self, component_dict):

        component_list = ''

        for i, (key, value) in enumerate(component_dict.items()):
            
            num = i + 1
            if value:
                component_list += f'{num}. {key}: {value["name"]} | price: {value["price"]}\n'
            else:
                component_list += f'{num}. {key}: None | price: None \n'

        return component_list
    
    def parser_message(self, message):
        pattern = {
            "reason": r"\[ 推薦原因 \] ：\s*(.*?)\s*\n",
            "name": r"\[ 產品名稱 \] ：\s*(.*?)\s*\n",
            "price": r"\[ 產品價格 \] ：\s*(\d+)"
        }

        results = {key: re.search(regex, message).group(1) for key, regex in pattern.items()}

        return results
    
class Reducing_Agent(Agent):
    
    def __init__(self, agent_name, prompt, model, require, budget):
        super().__init__(agent_name, prompt, model, require, budget)

    def action(self, component_dict):
        
        component_list = self.get_component_list(component_dict)
        total_price = self.summary_price(component_list)
        
        difference_price = self.budget - total_price
        
        if difference_price > 0:
            return False
        else: 
            user_message = {}
            user_message['component_list'] = component_list
            user_message['total_price'] = total_price
            user_message['difference_price'] = difference_price
            user_message['require'] = self.require
            user_message['budget'] = self.budget
            
            message = self.llm.invoke(user_message)
            self.memory.append(message)
        
            replace_component, results = self.parser_replace_message(message)
            component_dict[replace_component] = results
        
            return component_dict
    
    def summary_price(self, component_list):
        
        price_pattern = r"price: (\d+)"

        prices = [int(price) for price in re.findall(price_pattern, component_list)]

        return sum(prices)

    def parser_replace_message(self, message):
        
        patterns = {
            "replace_component": r"\[ 替換產品 \] ：\s*(.*?)\s*\n",
            "reason": r"\[ 推薦原因 \] ：\s*(.*?)\s*\n",
            "name": r"\[ 產品名稱 \] ：\s*(.*?)\s*\n",
            "price": r"\[ 產品價格 \] ：\s*(\d+)"
        }

        replace_component = re.search(patterns["replace_component"], message).group(1)

        results = {
            "reason": re.search(patterns["reason"], message).group(1),
            "name": re.search(patterns["name"], message).group(1),
            "price": re.search(patterns["price"], message).group(1)
        }

        return replace_component, results

def get_component_list(component_dict):

    component_list = ''

    for i, (key, value) in enumerate(component_dict.items()):
        
        num = i + 1
        if value:
            component_list += f'{num}. {key}: {value["name"]} | price: {value["price"]}\n'
        else:
            component_list += f'{num}. {key}: None | price: None \n'

    return component_list

def parser_message(message):
    pattern = {
        "reason": r"\[ 推薦原因 \] ：\s*(.*?)\s*\n",
        "name": r"\[ 產品名稱 \] ：\s*(.*?)\s*\n",
        "price": r"\[ 產品價格 \] ：\s*(\d+)"
    }

    results = {key: re.search(regex, message).group(1) for key, regex in pattern.items()}

    return results

def summary_price(component_list):
    price_pattern = r"price: (\d+)"

    prices = [int(price) for price in re.findall(price_pattern, component_list)]

    return sum(prices)

def parser_replace_message(message):
    
    patterns = {
        "replace_component": r"\[ 替換產品 \] ：\s*(.*?)\s*\n",
        "reason": r"\[ 推薦原因 \] ：\s*(.*?)\s*\n",
        "name": r"\[ 產品名稱 \] ：\s*(.*?)\s*\n",
        "price": r"\[ 產品價格 \] ：\s*(\d+)"
    }

    replace_component = re.search(patterns["replace_component"], message).group(1)

    results = {
        "reason": re.search(patterns["reason"], message).group(1),
        "name": re.search(patterns["name"], message).group(1),
        "price": re.search(patterns["price"], message).group(1)
    }

    return replace_component, results

def component_prompt_maker(component_name):

    prompt = f"""
    你是一個專業的電腦-{component_name}銷售機器人，請根據你的專業，
    根據客戶的需求，提供你覺得客戶可能需要的電腦-{component_name}給他。
    """
    
    prompt = prompt + """
    客戶的預算為：{budget}
    客戶的需求為：{require}
    目前已經有的電腦組合清單為：{component_list}

    請說明推薦原因，並提供產品名稱，範例的回饋訊息如下：
    [ 推薦原因 ] ： 耐用性強，可以使用到最新規格的Intel CPU...
    [ 產品名稱 ] ： Inter 第八代 i9
    [ 產品價格 ] ： 13000
    
    另外有幾點請注意：
    1. 推薦原因的說明請限制在50個字以內
    2. 產品的價格，請回覆一個完整的數字，中間不需要加入逗號
    """

    return prompt

def get_budget_from_string(text):
    
    text = text.split('。')[0] # 取出第一句話
    match = re.search(r'\d+', text) # 取數字
    if match:
        budget = int(match.group())
    else:
        print("Not found the number")
    
    return budget

def print_results(component_dict):
    component_list = get_component_list(component_dict)
    print(f'組合清單：{component_list}')

    total_price = summary_price(component_list)
    print(f'總價格：{total_price}')


In [3]:
# Ask for component list
model = 'gpt-4o-mini'

# 金額必須輸入為: xxx元，這句需求的文字陳述後，需要加上。
require = '現在有一個使用者想要30000元的遊戲機器。'
budget = get_budget_from_string(require)

### Component building ###

component_dict = {'Mother board': '', 'Case': '', 'CPU': '', 'GPU': '', 
                  'Memory': '', 'Device': '', 'Power': '', 'Fan': ''}
agent_dict = {}

for component_name in component_dict.keys():
    
    component_prompt = component_prompt_maker(component_name)
    agent = Agent(component_name, component_prompt, model, require, budget)
    agent_dict[component_name] = agent
    
### Reducing price strategy ###

reducing_prompt ="""
    你是一個專業的電腦銷售機器人，目前客戶遇到了一個難題，
    它有一組電腦的清單，清單中含有產品的名稱與價格，
    但清單的總價格超出了預算。
    
    身為專業的電腦銷售機器人，請把它挑出這個電腦清單中，
    哪一個產品可以被替換成更低價格的產品，從而使得整體的價格可以低於或者等於客戶的預算。
    
    客戶的需求為：{require}
    客戶的預算為：{budget}
    目前已經有的電腦組合清單為：{component_list}
    目前已經有的電腦組合清單的總價格：{total_price}
    距離客戶預算的差價：{difference_price}

    請指出哪一個種類的產品可以被替換，
    推薦的一個更低價格的替代方案。
    請說明推薦原因，並提供產品名稱，範例的回饋訊息如下：
    [ 替換產品 ] ： CPU
    [ 推薦原因 ] ： 耐用性強，可以使用到最新規格的Intel CPU...
    [ 產品名稱 ] ： Inter 第八代 i9
    [ 產品價格 ] ： 13000
    
    另外有幾點請注意：
    1. 推薦原因的說明請限制在50個字以內
    2. 產品的價格，請回覆一個完整的數字，中間不需要加入逗號
    """       

reducing_agent = Reducing_Agent('Reducing_agent', reducing_prompt, model, require, budget)

print('Mutli-agent initialization')

Mutli-agent initialization


In [4]:
# Find the component list
for agent_name, agent in agent_dict.items():
    
    component_dict = agent.action(component_dict)
    
print_results(component_dict)

組合清單：1. Mother board: ASUS ROG Strix B550-F Gaming | price: 5999
2. Case: Cooler Master MasterBox Q300L | price: 1890
3. CPU: AMD Ryzen 5 5600X | price: 6500
4. GPU: NVIDIA GeForce RTX 3060 | price: 13000
5. Memory: Corsair Vengeance LPX 16GB (2 x 8GB) DDR4 3200MHz | price: 2990
6. Device: Corsair RM650 650W 80+ Gold | price: 3500
7. Power: Cooler Master Hyper 212 RGB | price: 1290
8. Fan: Noctua NF-A14 PWM | price: 1200

總價格：36369


In [ ]:
# Reducing strategy
while True:
    response = reducing_agent.action(component_dict)
    if response:
        component_dict = response
    else:
        break
    print_results(component_dict)


組合清單：1. Mother board: ASUS ROG Strix B550-F Gaming | price: 5999
2. Case: Cooler Master MasterBox Q300L | price: 1890
3. CPU: AMD Ryzen 5 5600X | price: 6500
4. GPU: NVIDIA GeForce GTX 1660 Super | price: 8000
5. Memory: Corsair Vengeance LPX 16GB (2 x 8GB) DDR4 3200MHz | price: 2990
6. Device: Corsair RM650 650W 80+ Gold | price: 3500
7. Power: Cooler Master Hyper 212 RGB | price: 1290
8. Fan: Noctua NF-A14 PWM | price: 1200

總價格：31369
組合清單：1. Mother board: ASUS ROG Strix B550-F Gaming | price: 5999
2. Case: Cooler Master MasterBox Q300L | price: 1890
3. CPU: AMD Ryzen 5 5600X | price: 6500
4. GPU: NVIDIA GeForce GTX 1650 Super | price: 6000
5. Memory: Corsair Vengeance LPX 16GB (2 x 8GB) DDR4 3200MHz | price: 2990
6. Device: Corsair RM650 650W 80+ Gold | price: 3500
7. Power: Cooler Master Hyper 212 RGB | price: 1290
8. Fan: Noctua NF-A14 PWM | price: 1200

總價格：29369
